# S6E8 — EDA + baseline

Playground Series **S6E8, "Predicting Smartphone Addiction"**. Binary classification on
`addicted_label`, scored on **ROC AUC**, submission is a **probability**.

This notebook does two jobs and nothing else:

1. **EDA** — establish what the 12 features are, where the signal is, and whether train and test
   agree with each other.
2. **Baseline** — one raw-feature 5-fold LightGBM on the project's frozen CV split, emitting the OOF
   and test probability artifacts that every later run will be blended against.

No feature engineering, no tuning, no ensembling. The point of this run is to produce a *trustworthy
first measurement* — the first (OOF, LB) pair — not a competitive score. See `README.md` for the
frozen-split contract and `KAGGLE_PLAYBOOK.md` for why that ordering matters.

In [ ]:
import json, os, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
import lightgbm as lgb

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

T_START = time.time()

# ---- the frozen contract (README.md). Do not change these. -----------------
SEED      = 42
N_FOLDS   = 5
TARGET    = "addicted_label"
ID        = "id"
COMP      = "playground-series-s6e8"

# ---- dual path: same notebook runs as a Kaggle kernel and locally ----------
KAGGLE_DIR = Path("/kaggle/input/competitions") / COMP
ON_KAGGLE  = KAGGLE_DIR.exists()
if ON_KAGGLE:
    DATA_DIR = KAGGLE_DIR
    OUT_DIR  = Path("/kaggle/working")
else:
    DATA_DIR = Path("data")
    OUT_DIR  = Path("experiments/preds/local")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"on_kaggle={ON_KAGGLE}  data={DATA_DIR}  out={OUT_DIR}")
print(f"lightgbm {lgb.__version__} | pandas {pd.__version__} | numpy {np.__version__}")

## 1. Load and schema

The first thing to nail down is the `RAW_NUM` / `RAW_CAT` split. Every downstream notebook is built
around those two explicit lists, so they get declared here once and printed, rather than being
re-inferred ad hoc later.

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
test  = pd.read_csv(DATA_DIR / "test.csv")
sub   = pd.read_csv(DATA_DIR / "sample_submission.csv")

print(f"train {train.shape}   test {test.shape}   sample_submission {sub.shape}")
print(f"train-only columns : {sorted(set(train.columns) - set(test.columns))}")
print(f"test-only columns  : {sorted(set(test.columns) - set(train.columns))}")
print(f"train id range: {train[ID].min()}..{train[ID].max()}   "
      f"test id range: {test[ID].min()}..{test[ID].max()}   "
      f"overlap: {len(set(train[ID]) & set(test[ID]))}")
print(f"memory: train {train.memory_usage(deep=True).sum()/1e6:.0f} MB   "
      f"test {test.memory_usage(deep=True).sum()/1e6:.0f} MB")
train.head()

In [ ]:
# The two lists the whole project is built around.
FEATURES = [c for c in train.columns if c not in (ID, TARGET)]
RAW_CAT  = [c for c in FEATURES if train[c].dtype == object]
RAW_NUM  = [c for c in FEATURES if c not in RAW_CAT]

schema = pd.DataFrame({
    "dtype":     train[FEATURES].dtypes.astype(str),
    "kind":      ["cat" if c in RAW_CAT else "num" for c in FEATURES],
    "nunique":   train[FEATURES].nunique(),
    "n_null_tr": train[FEATURES].isna().sum(),
    "pct_null_tr": (train[FEATURES].isna().mean() * 100).round(2),
    "pct_null_te": (test[FEATURES].isna().mean() * 100).round(2),
})
print(f"{len(RAW_NUM)} numeric + {len(RAW_CAT)} categorical = {len(FEATURES)} features\n")
print(f"RAW_NUM = {RAW_NUM}")
print(f"RAW_CAT = {RAW_CAT}\n")
schema

## 2. Target

`sample_submission.csv` is a constant column. If that constant equals the train positive rate, it
confirms the target definition and gives a free sanity check that we are reading the right column.

In [ ]:
pos_rate = train[TARGET].mean()
sub_const = sub[TARGET].iloc[0]
print(f"train positive rate        : {pos_rate:.6f}   ({train[TARGET].sum():,} of {len(train):,})")
print(f"sample_submission constant : {sub_const:.6f}   (unique values: {sub[TARGET].nunique()})")
print(f"match: {np.isclose(pos_rate, sub_const, atol=1e-4)}")
print()
print("The POSITIVE class is the majority (71/29). Imbalance is mild, and AUC is rank-based,")
print("so no resampling / class weighting is warranted by default -- it is a probe, not a given.")

## 3. Missingness

Every column is missing 4–19% of its values, and the rates are visibly *different* between train and
test. That difference is the kind of thing that is either a genuine trap or nothing at all, so it is
worth resolving now rather than in week three.

The decisive test: **does the missingness itself predict the target?** If a missing-indicator column
carries signal, missingness is informative (MNAR) and imputation strategy matters a lot. If every
indicator sits at AUC 0.500, the values were blanked at random and the indicators are pure noise.

In [ ]:
rows = []
for c in FEATURES:
    na_tr = train[c].isna()
    rows.append({
        "feature": c,
        "pct_null_train": na_tr.mean() * 100,
        "pct_null_test":  test[c].isna().mean() * 100,
        "delta_pp":       (test[c].isna().mean() - na_tr.mean()) * 100,
        # AUC of the missing-indicator alone: 0.5 == missingness carries no signal
        "auc_missing_ind": roc_auc_score(train[TARGET], na_tr.astype(int)),
    })
miss = pd.DataFrame(rows).sort_values("pct_null_train", ascending=False)
miss.round(4)

In [ ]:
worst = (miss["auc_missing_ind"] - 0.5).abs().max()
print(f"largest |AUC(missing-indicator) - 0.5| across all {len(FEATURES)} features: {worst:.5f}")
print()
if worst < 0.01:
    print("VERDICT: missingness is uninformative (MCAR by construction).")
    print("  -> missing-indicator features are noise; do not add them.")
    print("  -> the train/test rate gaps are sampling noise, not a distribution shift.")
    print("  -> LightGBM's native NaN handling is sufficient; no imputation needed for the baseline.")
else:
    print("VERDICT: at least one missing-indicator carries signal -- investigate before imputing.")

# How much of a row is missing at once? Matters for whether row-wise imputation is even viable.
nmiss = train[FEATURES].isna().sum(axis=1)
print(f"\nmissing values per row: mean {nmiss.mean():.2f} of {len(FEATURES)}, "
      f"max {nmiss.max()}, fully-complete rows {(nmiss == 0).mean()*100:.1f}%")
print(f"AUC of 'number of missing fields in this row': "
      f"{roc_auc_score(train[TARGET], nmiss):.5f}")

## 4. Train ↔ test drift

Playground data is synthetic and usually well matched, but confirming it is cheap and the failure
mode is expensive: a shifted column means OOF cannot predict the LB no matter how clean the CV is.

Two-sample KS for numerics, chi-square on the category distribution for categoricals. With ~691k vs
~296k rows, *any* real difference is statistically significant, so read the **effect size** (the KS
statistic itself) rather than the p-value.

In [ ]:
rows = []
for c in RAW_NUM:
    a, b = train[c].dropna(), test[c].dropna()
    ks = stats.ks_2samp(a, b)
    rows.append({"feature": c, "kind": "num", "stat": ks.statistic, "pvalue": ks.pvalue,
                 "mean_tr": a.mean(), "mean_te": b.mean()})
for c in RAW_CAT:
    ct = pd.concat([train[c].value_counts(), test[c].value_counts()], axis=1, keys=["tr", "te"]).fillna(0)
    chi2, p, _, _ = stats.chi2_contingency(ct.T.values)
    # Cramer's V as a comparable effect size
    n = ct.values.sum()
    v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
    rows.append({"feature": c, "kind": "cat", "stat": v, "pvalue": p,
                 "mean_tr": np.nan, "mean_te": np.nan})

drift = pd.DataFrame(rows).sort_values("stat", ascending=False)
print("stat = KS statistic (num) or Cramer's V (cat). Both are 0..1; <0.01 is negligible.\n")
print(drift.round(5).to_string(index=False))
print(f"\nmax effect size across all features: {drift['stat'].max():.5f}")
print("VERDICT:", "no meaningful train/test drift." if drift["stat"].max() < 0.02
      else "a feature is shifted -- investigate before trusting OOF.")

## 5. Where the signal is

The cheapest possible read on the problem: score each feature *alone* as a ranker. For a numeric
column that is `roc_auc_score(y, x)` directly (AUC is rank-based, so no model is needed). For a
categorical, map each level to its train positive rate and score that.

A single-feature AUC of 0.50 means no marginal signal; 1.0 means the feature alone solves it. Values
below 0.50 are *inverted* signal, which is just as useful — the distance from 0.50 is what matters.

Note this measures **marginal** signal only. A feature flat at 0.50 alone can still matter in
interaction, so this ranking is a map of the terrain, not a feature-selection decision.

In [ ]:
rows = []
for c in RAW_NUM:
    m = train[c].notna()
    auc = roc_auc_score(train.loc[m, TARGET], train.loc[m, c])
    rows.append({"feature": c, "kind": "num", "solo_auc": auc, "abs_lift": abs(auc - 0.5)})
for c in RAW_CAT:
    m = train[c].notna()
    rate = train.loc[m].groupby(c)[TARGET].mean()
    auc = roc_auc_score(train.loc[m, TARGET], train.loc[m, c].map(rate))
    rows.append({"feature": c, "kind": "cat", "solo_auc": auc, "abs_lift": abs(auc - 0.5)})

solo = pd.DataFrame(rows).sort_values("abs_lift", ascending=False).reset_index(drop=True)
print(solo.round(4).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
colors = ["#2b7bba" if k == "num" else "#d1741f" for k in solo["kind"]]
ax.barh(solo["feature"][::-1], (solo["solo_auc"] - 0.5)[::-1], color=colors[::-1])
ax.axvline(0, color="k", lw=1)
ax.set_xlabel("single-feature AUC − 0.50   (distance from 0 = marginal signal)")
ax.set_title("Marginal signal per feature (blue = numeric, orange = categorical)")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Per-level positive rates for the categoricals -- the three-level spread tells us
# directly how much a categorical could ever contribute.
for c in RAW_CAT:
    g = train.groupby(c)[TARGET].agg(["count", "mean"])
    g["mean"] = g["mean"].round(4)
    g["vs_base_pp"] = ((g["mean"] - pos_rate) * 100).round(2)
    print(f"--- {c}  (null: {train[c].isna().mean()*100:.1f}%)")
    print(g.to_string(), "\n")

In [ ]:
# Distribution of the top numeric features, split by target class. This is where the
# shape of the decision boundary becomes visible.
top_num = [f for f in solo.query("kind == 'num'")["feature"][:6]]
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, c in zip(axes.ravel(), top_num):
    for cls, color in [(0, "#4c9f70"), (1, "#c0392b")]:
        ax.hist(train.loc[train[TARGET] == cls, c].dropna(), bins=60, density=True,
                alpha=0.55, color=color, label=f"{TARGET}={cls}")
    ax.set_title(f"{c}  (solo AUC {solo.set_index('feature').loc[c, 'solo_auc']:.3f})", fontsize=10)
    ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 6. Redundancy among the numerics

Two things to look for: near-duplicate columns (|r| > 0.95, which would make one of them dead
weight), and *blocks* of correlated features, which are where engineered ratios and differences tend
to pay off later.

In [ ]:
corr = train[RAW_NUM].corr()
fig, ax = plt.subplots(figsize=(8, 6.5))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(RAW_NUM))); ax.set_xticklabels(RAW_NUM, rotation=90, fontsize=8)
ax.set_yticks(range(len(RAW_NUM))); ax.set_yticklabels(RAW_NUM, fontsize=8)
for i in range(len(RAW_NUM)):
    for j in range(len(RAW_NUM)):
        if i != j and abs(corr.iloc[i, j]) > 0.3:
            ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=7)
fig.colorbar(im, ax=ax, shrink=0.8)
ax.set_title("Pearson correlation, numeric features (pairwise-complete)")
plt.tight_layout(); plt.show()

pairs = [(RAW_NUM[i], RAW_NUM[j], corr.iloc[i, j])
         for i in range(len(RAW_NUM)) for j in range(i + 1, len(RAW_NUM))]
pairs.sort(key=lambda t: -abs(t[2]))
print("strongest pairs:")
for a, b, r in pairs[:8]:
    flag = "  <-- NEAR-DUPLICATE" if abs(r) > 0.95 else ""
    print(f"  {a:<26} {b:<26} r={r:+.4f}{flag}")

## 7. Baseline — raw-feature 5-fold LightGBM

Deliberately boring. No feature engineering, no tuning, near-default parameters. Its only jobs are to
produce an honest OOF number and the probability artifacts that every future run gets blended
against.

**Encoding.** Categoricals become pandas `category` dtype with the vocabulary taken from
`train ∪ test`, which LightGBM consumes natively. This is an *unsupervised* label mapping — it never
touches `y` — so it is not a leak, unlike target/count encoding, which must be fit per fold (see
`README.md`). NaNs are left alone; LightGBM routes them natively at each split.

In [ ]:
# Unsupervised category vocabulary over train u test -- never touches y, so not a leak.
X      = train[FEATURES].copy()
X_test = test[FEATURES].copy()
for c in RAW_CAT:
    levels = pd.Index(sorted(set(train[c].dropna()) | set(test[c].dropna())))
    X[c]      = pd.Categorical(X[c],      categories=levels)
    X_test[c] = pd.Categorical(X_test[c], categories=levels)
y = train[TARGET].values

print("dtypes fed to LightGBM:")
print(X.dtypes.to_string())
assert list(X.columns) == list(X_test.columns), "train/test column order must match"

In [ ]:
LGB_PARAMS = dict(
    objective="binary",
    # Deliberately generous: early stopping, not this cap, must decide the tree count.
    # At 3000 one fold hit the cap and never stopped, which makes that fold's model
    # under-trained and the anchor dishonest. A cap that binds is a silent defect.
    n_estimators=8000,
    learning_rate=0.05,
    num_leaves=31,          # library default
    random_state=SEED,
    n_jobs=-1,
    verbose=-1,
)
EARLY_STOP = 100

# THE frozen split (README.md). Never change these three arguments.
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

oof        = np.zeros(len(X))
fold_id    = np.full(len(X), -1, dtype=int)
test_proba = np.zeros(len(X_test))
fold_aucs, best_iters = [], []

for k, (itr, iva) in enumerate(skf.split(X, y)):
    fold_id[iva] = k
    model = lgb.LGBMClassifier(**LGB_PARAMS)
    model.fit(
        X.iloc[itr], y[itr],
        eval_set=[(X.iloc[iva], y[iva])],
        eval_metric="auc",
        callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False)],
    )
    oof[iva] = model.predict_proba(X.iloc[iva])[:, 1]
    test_proba += model.predict_proba(X_test)[:, 1] / N_FOLDS
    auc = roc_auc_score(y[iva], oof[iva])
    fold_aucs.append(auc); best_iters.append(int(model.best_iteration_))
    print(f"  fold {k}: n_train={len(itr):,} n_val={len(iva):,}  "
          f"AUC={auc:.6f}  best_iter={model.best_iteration_}")

oof_auc = roc_auc_score(y, oof)
print(f"\nOOF AUC (pooled)  : {oof_auc:.6f}")
print(f"fold mean +/- std : {np.mean(fold_aucs):.6f} +/- {np.std(fold_aucs):.6f}")
print(f"fold spread       : {max(fold_aucs) - min(fold_aucs):.6f}")
print("\nNOTE: fold spread measures EVALUATION-FOLD DIFFICULTY, not test-prediction variance.")
print("      Test predictions are already averaged over all 5 fold-models. Do not conflate the two")
print("      and go spend GPU-hours re-running at a new split seed (KAGGLE_PLAYBOOK.md section 5).")

In [ ]:
# assert the frozen split really is the frozen split -- cheap, and catches a whole class of
# silent cross-run misalignment later.
assert skf.n_splits == 5 and skf.shuffle is True and skf.random_state == 42, "CV split drifted!"
assert (fold_id >= 0).all(), "every row must be assigned exactly one validation fold"
# early stopping, not the n_estimators cap, must decide the tree count -- a fold that
# hits the cap is under-trained, and silently so.
assert max(best_iters) < LGB_PARAMS["n_estimators"], (
    f"a fold hit the n_estimators cap ({max(best_iters)}); raise it -- this run is under-trained")
print("fold sizes:", np.bincount(fold_id))
print("per-fold positive rate:", [round(y[fold_id == k].mean(), 5) for k in range(N_FOLDS)])
print(f"best_iter max {max(best_iters)} of cap {LGB_PARAMS['n_estimators']} -- early stopping bound on every fold")

In [ ]:
# Split-count importance vs the MARGINAL AUC from section 5. Where these disagree is the
# most informative output in this notebook: a feature the model leans on heavily but that
# ranks flat on its own is contributing only through INTERACTIONS -- which is exactly what
# a univariate ranking is blind to, and exactly where feature engineering has room.
imp = (pd.Series(model.feature_importances_, index=FEATURES) /
       model.feature_importances_.sum() * 100)
cmp = (solo.set_index("feature")
           .assign(split_pct=imp)
           .sort_values("split_pct", ascending=False)
           [["kind", "solo_auc", "abs_lift", "split_pct"]])
cmp["marginal_rank"] = cmp["abs_lift"].rank(ascending=False).astype(int)
cmp["usage_rank"]    = cmp["split_pct"].rank(ascending=False).astype(int)
cmp["rank_gap"]      = cmp["marginal_rank"] - cmp["usage_rank"]
print("feature importance (last fold, % of splits) vs marginal signal:\n")
print(cmp.round(4).to_string())
print("\nrank_gap > 0 => the model uses it MORE than its solo signal justifies (interaction-only).")

## 8. Artifacts

The highest-ROI thing this notebook does (`KAGGLE_PLAYBOOK.md` §1): save the per-learner OOF and test
probabilities keyed by row id, for **every** run. That is what makes any past model retro-blendable
with any future model without retraining, and it costs nothing now.

The OOF file carries its `fold` column so any future run can verify it was built on the same split
instead of assuming it.

In [ ]:
LEARNER = "lgb"

pd.DataFrame({ID: train[ID], "fold": fold_id, "proba": oof}) \
  .to_csv(OUT_DIR / f"oof_proba_{LEARNER}.csv", index=False)
pd.DataFrame({ID: test[ID], "proba": test_proba}) \
  .to_csv(OUT_DIR / f"test_proba_{LEARNER}.csv", index=False)

submission = pd.DataFrame({ID: test[ID], TARGET: test_proba})
submission.to_csv(OUT_DIR / "submission.csv", index=False)

# ---- validate against sample_submission BEFORE trusting it ----------------
assert list(submission.columns) == [ID, TARGET], f"bad columns: {list(submission.columns)}"
assert len(submission) == len(sub), f"row count {len(submission)} != {len(sub)}"
assert (submission[ID].values == sub[ID].values).all(), "ids must match sample_submission in ORDER"
assert submission[TARGET].notna().all(), "NaN in submission"
assert submission[TARGET].between(0, 1).all(), "probabilities outside [0,1]"
print("submission validated against sample_submission.csv")
print(submission[TARGET].describe().round(5).to_string())
print()
print(submission.head())

In [ ]:
RUN_METRICS = {
    "final_oof_auc":   round(float(oof_auc), 6),
    f"{LEARNER}_oof_auc": round(float(oof_auc), 6),
    "fold_aucs":       [round(float(a), 6) for a in fold_aucs],
    "fold_auc_mean":   round(float(np.mean(fold_aucs)), 6),
    "fold_auc_std":    round(float(np.std(fold_aucs)), 6),
    "best_iters":      best_iters,
    "n_features":      len(FEATURES),
    "n_train":         int(len(train)),
    "n_test":          int(len(test)),
    "n_folds":         N_FOLDS,
    "cv_seed":         SEED,
    "learners":        [LEARNER],
    "notebook_runtime_sec": round(time.time() - T_START, 1),
}
# scripts/collect_run.py parses exactly this line out of the kernel log.
print("RUN_METRICS_JSON:" + json.dumps(RUN_METRICS))

## 9. Findings and next probes

**What the EDA established**

- 12 features: 9 numeric, 3 categorical. Small, dense, fully synthetic-looking. No id leakage
  (train and test ids are disjoint and contiguous).
- **Missingness is uninformative.** Every missing-indicator scores AUC ≈ 0.500. So: missing-indicator
  features are noise, the train/test missingness-rate gaps are not a distribution shift, and
  LightGBM's native NaN routing is enough — imputation is a probe, not a requirement.
- **No meaningful train/test drift** on any column. OOF should track the LB; if it doesn't, the bug
  is ours.
- **Marginal signal is concentrated in screen-time.** `daily_screen_time_hours` (solo AUC 0.890),
  `weekend_screen_time` (0.881) and `social_media_hours` (0.858) carry nearly all of it;
  `work_study_hours` (0.655) and `gaming_hours` (0.622) carry a little. `age` (0.502),
  `sleep_hours` (0.527), `app_opens_per_day` (0.541), `notifications_per_day` (0.492 — *inverted*)
  and **all three categoricals** (≤0.512) are marginally flat.

**But the model's usage contradicts that ranking, and that is the real finding.** Section 7's
  split-count importance puts `notifications_per_day` and `app_opens_per_day` **first and second** —
  above every screen-time column — despite both being marginally flat. They cannot be noise: a
  tree model does not spend a quarter of its splits on noise. They are **interaction-only**
  features, which is precisely the thing a univariate AUC ranking cannot see. Any feature selection
  driven by marginal signal alone would have thrown away the two columns the model leans on hardest.

**Baseline result.** OOF AUC **0.9633** (folds 0.9626–0.9642, spread 0.0016) from raw features and
  near-default parameters. Public LB currently tops out around 0.9709, so a no-feature-engineering
  baseline is already within ~0.008 of the leaders — the remaining headroom is small and will have to
  be earned.

**Candidate probes, in the order they look most promising.** Each needs a written hypothesis and a
pre-registered gate before it runs (`KAGGLE_PLAYBOOK.md` §3) — and the gate itself cannot be set
until §4's OOF↔LB residual σ has been measured over ~10 runs. Nothing here is licensed to ship yet.

1. **Explicit interaction terms for the two interaction-only features.** The split-count/marginal-AUC
   inversion says the model is spending most of its capacity *discovering* how
   `notifications_per_day` and `app_opens_per_day` combine with screen time. Handing it those
   combinations directly — `notifications / app_opens` (notifications per open),
   `app_opens / daily_screen_time` (opens per hour), `notifications / daily_screen_time` — is the
   best-motivated probe on the board, because it is grounded in a measured contradiction rather than
   a guess.
2. **Composition ratios.** `daily_screen_time_hours` and its plausible components
   (`social_media_hours`, `gaming_hours`, `work_study_hours`) may form a whole/parts relationship:
   the component means (2.47 + 1.46 + 2.37 = 6.30) sit just under the total mean (7.64). Shares, the
   unexplained residual `daily_screen_time − (social + gaming + work_study)`, and the
   weekend/weekday ratio are the obvious constructions. Unsupervised, so cheap and leak-free.
3. **Screen-time vs sleep displacement.** `sleep_hours` is marginally flat and mid-pack in split
   usage — another interaction candidate, and the natural denominator for a displacement feature.
4. **The flat categoricals.** Three levels each, ±0.2pp on the positive rate, and together <3% of
   splits. Likely noise columns planted by the generator. Confirm by ablation rather than assuming —
   but expect nothing, and do not build target encoding for them on hope.
5. **Original source dataset.** Playground data is generated from a real dataset; if the source is
   findable, appending it as extra training rows is a standard Playground lever. Worth one probe.

**Caveats to carry forward.** Early stopping selects the tree count on the same fold that is then
scored, so the OOF is very slightly optimistic — a constant bias across runs, so it does not distort
*comparisons*, which is what the gate is measured on. And per-fold AUC spread (0.0016) is
evaluation-fold difficulty, not test-prediction variance.

**Explicitly deferred:** HPO, alternative learners, neural legs, and any blending. Per the playbook,
none of that is worth doing until the measuring instrument — `experiments/runs.csv` with enough
paired OOF/LB rows to set a gate — actually exists.